# PhoBERT Baseline - Cross Entropy Loss (Kaggle)

## Mô tả
Notebook này triển khai mô hình PhoBERT baseline cho phân tích cảm xúc sinh viên.

**Đặc điểm:**
- Sử dụng PhoBERT pre-trained (vinai/phobert-base)
- Loss function: CrossEntropyLoss (standard)
- Không sử dụng data augmentation
- Không sử dụng class weights
- **Epochs: 10, Early Stop: 2**

**Mục đích:** Làm baseline để so sánh với các phương pháp cải tiến.

## 1. Setup và Import Libraries

In [2]:
import os
import sys
import random
import copy
import re
from pathlib import Path
from datetime import datetime
from collections import Counter


def find_kaggle_project_root():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        raise FileNotFoundError('/kaggle/input not found')

    # Preferred structure: /kaggle/input/<dataset>/data/processed
    for ds in input_root.iterdir():
        if ds.is_dir() and (ds / 'data' / 'processed').exists():
            return ds

    # Fallback: deep scan for data/processed
    for processed_dir in input_root.rglob('processed'):
        if processed_dir.is_dir() and processed_dir.parent.name == 'data':
            return processed_dir.parent.parent

    raise FileNotFoundError(
        'Cannot find Kaggle dataset root containing data/processed. '
        f'Available folders: {list(input_root.glob("*"))}'
    )


PROJECT_ROOT = find_kaggle_project_root()
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Kaggle inputs = {list(Path("/kaggle/input").glob("*"))}')

PROJECT_ROOT = /kaggle/input/datasets/jadt145/phobert
Kaggle inputs = [PosixPath('/kaggle/input/datasets')]


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score, precision_recall_fscore_support
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Configuration

In [ ]:
class Config:
    BASE_DIR = str(PROJECT_ROOT)
    DATA_DIR = os.path.join(BASE_DIR, 'data', 'processed')

    MODEL_TYPE = 'PhoBERT_Baseline'
    EXPERIMENT_TYPE = 'improvements'
    TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

    RESULTS_ROOT = '/kaggle/working'
    RESULTS_DIR = os.path.join(RESULTS_ROOT, 'results', MODEL_TYPE, EXPERIMENT_TYPE, TIMESTAMP)
    MODELS_DIR = os.path.join(RESULTS_DIR, 'models')
    SUMMARIES_DIR = os.path.join(RESULTS_DIR, 'summaries')
    VISUALIZATIONS_DIR = os.path.join(RESULTS_DIR, 'visualizations')
    ARTIFACTS_DIR = os.path.join(RESULTS_DIR, 'artifacts')

    MODEL_NAME = 'vinai/phobert-base'
    MAX_LENGTH = 256
    BATCH_SIZE = 8  # Reduced from 16 to avoid OOM
    GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch size = 8 * 2 = 16
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 10
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    EARLY_STOPPING_PATIENCE = 2
    NUM_CLASSES = 3
    LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    USE_FP16 = True  # Mixed precision training


config = Config()
# Create all subdirectories
for dir_path in [config.RESULTS_DIR, config.MODELS_DIR, config.SUMMARIES_DIR,
                 config.VISUALIZATIONS_DIR, config.ARTIFACTS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('=' * 60)
print('PHOBERT BASELINE - KAGGLE VERSION')
print('=' * 60)
print(f'DATA_DIR: {config.DATA_DIR}')
print(f'Results will be saved to: {config.RESULTS_DIR}')
print(f'Device: {config.DEVICE}')
print(f'Batch Size: {config.BATCH_SIZE}')
print(f'Gradient Accumulation Steps: {config.GRADIENT_ACCUMULATION_STEPS}')
print(f'Effective Batch Size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}')
print(f'Mixed Precision (FP16): {config.USE_FP16}')
print(f'Epochs: {config.NUM_EPOCHS}')
print(f'Early Stop Patience: {config.EARLY_STOPPING_PATIENCE}')

## 3. Load Data

In [5]:
def load_data(data_dir, split):
    split_dir = os.path.join(data_dir, split)
    with open(os.path.join(split_dir, 'sents.txt'), 'r', encoding='utf-8') as f:
        texts = [line.strip() for line in f.readlines()]
    with open(os.path.join(split_dir, 'sentiments.txt'), 'r', encoding='utf-8') as f:
        labels = [int(line.strip()) for line in f.readlines()]
    print(f'{split}: {len(texts)} samples')
    return texts, labels


train_texts, train_labels = load_data(config.DATA_DIR, 'train')
val_texts, val_labels = load_data(config.DATA_DIR, 'validation')
test_texts, test_labels = load_data(config.DATA_DIR, 'test')

train: 11426 samples
validation: 1583 samples
test: 3166 samples


In [6]:
print('Label distribution:')
for split_name, labels in [('Train', train_labels), ('Val', val_labels), ('Test', test_labels)]:
    counter = Counter(labels)
    print(f'{split_name}:')
    for label, count in sorted(counter.items()):
        print(f'  {config.LABEL_MAP[label]}: {count} ({count/len(labels)*100:.1f}%)')

Label distribution:
Train:
  Negative: 5325 (46.6%)
  Neutral: 458 (4.0%)
  Positive: 5643 (49.4%)
Val:
  Negative: 705 (44.5%)
  Neutral: 73 (4.6%)
  Positive: 805 (50.9%)
Test:
  Negative: 1409 (44.5%)
  Neutral: 167 (5.3%)
  Positive: 1590 (50.2%)


## 4. Dataset và DataLoader

In [7]:
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
print(f'Tokenizer loaded: {config.MODEL_NAME}')

Tokenizer loaded: vinai/phobert-base


In [8]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [9]:
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, config.MAX_LENGTH)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer, config.MAX_LENGTH)
test_dataset = SentimentDataset(test_texts, test_labels, tokenizer, config.MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

Train batches: 715
Val batches: 99
Test batches: 198


## 5. Model Definition

In [10]:
class PhoBERTClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.1):
        super(PhoBERTClassifier, self).__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

In [ ]:
model = PhoBERTClassifier(model_name=config.MODEL_NAME, num_classes=config.NUM_CLASSES)
model = model.to(config.DEVICE)

# Clear cache after loading model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

if torch.cuda.is_available():
    print(f'GPU Memory after model load: {torch.cuda.memory_allocated() / 1024**2:.2f} MB')

## 6. Training Setup - CrossEntropyLoss (Baseline)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

# Account for gradient accumulation in scheduler
total_steps = len(train_loader) // config.GRADIENT_ACCUMULATION_STEPS * config.NUM_EPOCHS
warmup_steps = int(total_steps * config.WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

print('Loss function: CrossEntropyLoss (standard)')
print(f'Total training steps: {total_steps}')
print(f'Warmup steps: {warmup_steps}')
print(f'Gradient Accumulation Steps: {config.GRADIENT_ACCUMULATION_STEPS}')

## 7. Training Functions

In [ ]:
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler(enabled=config.USE_FP16)

def train_epoch(model, dataloader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    progress_bar = tqdm(dataloader, desc='Training')
    optimizer.zero_grad()
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        # Mixed precision forward pass
        with autocast(enabled=config.USE_FP16):
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels) / config.GRADIENT_ACCUMULATION_STEPS
        
        # Mixed precision backward pass
        scaler.scale(loss).backward()
        
        # Gradient accumulation
        if (step + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
        
        total_loss += loss.item() * config.GRADIENT_ACCUMULATION_STEPS
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        progress_bar.set_postfix({'loss': f'{loss.item() * config.GRADIENT_ACCUMULATION_STEPS:.4f}'})
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, accuracy, f1, f1_macro

In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating', leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            with autocast(enabled=config.USE_FP16):
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1_weighted = f1_score(all_labels, all_preds, average='weighted')
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    precision_pc, recall_pc, f1_pc, _ = precision_recall_fscore_support(all_labels, all_preds, labels=[0, 1, 2], zero_division=0)
    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_weighted': f1_weighted,
        'f1_macro': f1_macro,
        'f1_per_class': f1_pc.tolist(),
        'y_pred': list(map(int, all_preds)),
        'y_true': list(map(int, all_labels))
    }

## 8. Training Loop

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'train_f1': [], 'train_f1_macro': [],
           'val_loss': [], 'val_acc': [], 'val_f1': [], 'val_f1_macro': [], 'val_f1_neutral': []}
best_val_f1 = 0
best_epoch = 0
best_model_state = None
patience_counter = 0

# Clear GPU cache before training
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB')
    print(f'GPU Memory reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB')

print('='*60)
print('Starting Training - PhoBERT Baseline (CrossEntropyLoss)')
print('='*60)

for epoch in range(config.NUM_EPOCHS):
    print(f'\nEpoch {epoch + 1}/{config.NUM_EPOCHS}')
    train_loss, train_acc, train_f1, train_f1_macro = train_epoch(model, train_loader, criterion, optimizer, scheduler, config.DEVICE)
    val_results = evaluate(model, val_loader, criterion, config.DEVICE)
    val_f1 = val_results['f1_weighted']

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['train_f1_macro'].append(train_f1_macro)
    history['val_loss'].append(val_results['loss'])
    history['val_acc'].append(val_results['accuracy'])
    history['val_f1'].append(val_f1)
    history['val_f1_macro'].append(val_results['f1_macro'])
    history['val_f1_neutral'].append(val_results['f1_per_class'][1])

    print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}, F1_macro: {train_f1_macro:.4f}")
    print(f"Val   - Loss: {val_results['loss']:.4f}, Acc: {val_results['accuracy']:.4f}, F1: {val_f1:.4f}, F1_macro: {val_results['f1_macro']:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch + 1
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        torch.save(best_model_state, os.path.join(config.MODELS_DIR, 'phobert_model.pt'))
        print(f'  -> New best model saved! Val F1: {val_f1:.4f}')
    else:
        patience_counter += 1
        print(f'  -> No improvement. Patience: {patience_counter}/{config.EARLY_STOPPING_PATIENCE}')
        if patience_counter >= config.EARLY_STOPPING_PATIENCE:
            print(f'  -> Early stopping triggered at epoch {epoch + 1}')
            break
    
    # Clear cache after each epoch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Load best model
assert best_model_state is not None
model.load_state_dict(best_model_state)

print('='*60)
print(f'Training completed! Best Val F1: {best_val_f1:.4f} at epoch {best_epoch}')
print('='*60)

## 9. Training Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Validation')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(history['train_f1'], label='Train')
axes[2].plot(history['val_f1'], label='Validation')
axes[2].set_title('F1 Score over Epochs')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('F1 Score')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(config.VISUALIZATIONS_DIR, 'training_history.png'), dpi=150)
plt.show()

## 10. Evaluation on Test Set

In [ ]:
test_results = evaluate(model, test_loader, criterion, config.DEVICE)

print('='*60)
print('TEST SET RESULTS - PhoBERT Baseline (CrossEntropyLoss)')
print('='*60)
print(f'Loss: {test_results["loss"]:.4f}')
print(f'Accuracy: {test_results["accuracy"]:.4f}')
print(f'Precision (weighted): {test_results["precision"]:.4f}')
print(f'Recall (weighted): {test_results["recall"]:.4f}')
print(f'F1 Score (weighted): {test_results["f1_weighted"]:.4f}')
print(f'F1 Score (macro): {test_results["f1_macro"]:.4f}')
print(f'F1 Neutral: {test_results["f1_per_class"][1]:.4f}')

In [ ]:
print('Classification Report:')
target_names = [config.LABEL_MAP[i] for i in range(config.NUM_CLASSES)]
print(classification_report(test_results['y_true'], test_results['y_pred'], target_names=target_names, digits=4))

In [ ]:
cm = confusion_matrix(test_results['y_true'], test_results['y_pred'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix - PhoBERT Baseline')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(config.VISUALIZATIONS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## 11. Save Results

In [ ]:
# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(config.SUMMARIES_DIR, 'training_history.csv'), index=False)

# Save summary
summary = {
    'Model': 'PhoBERT Baseline',
    'Loss Function': 'CrossEntropyLoss',
    'Data Augmentation': 'No',
    'Class Weights': 'No',
    'Epochs Trained': len(history['train_loss']),
    'Best Epoch': best_epoch,
    'Best Val F1': best_val_f1,
    'Test Accuracy': test_results['accuracy'],
    'Test F1 (weighted)': test_results['f1_weighted'],
    'Test F1 (macro)': test_results['f1_macro'],
    'Test F1 Neutral': test_results['f1_per_class'][1],
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(os.path.join(config.SUMMARIES_DIR, 'summary.csv'), index=False)

# Save training results text
with open(os.path.join(config.SUMMARIES_DIR, 'training_results.txt'), 'w', encoding='utf-8') as f:
    f.write('=' * 60 + '\n')
    f.write('TRAINING RESULTS - PhoBERT Baseline\n')
    f.write('=' * 60 + '\n')
    f.write(f'Best Epoch: {best_epoch}\n')
    f.write(f'Test Accuracy: {test_results["accuracy"]:.4f}\n')
    f.write(f'Test F1 Macro: {test_results["f1_macro"]:.4f}\n')
    f.write(f'Test F1 Neutral: {test_results["f1_per_class"][1]:.4f}\n')

print('='*60)
print('SAVING RESULTS')
print('='*60)
print('Results Summary:')
for key, value in summary.items():
    if isinstance(value, float):
        print(f'{key}: {value:.4f}')
    else:
        print(f'{key}: {value}')

print(f'\nAll results saved to: {config.RESULTS_DIR}')
print('  - models/phobert_model.pt')
print('  - summaries/summary.csv, training_results.txt, training_history.csv')
print('  - visualizations/training_history.png, confusion_matrix.png')

## 12. Summary

### PhoBERT Baseline Model (Kaggle Version)

**Configuration:**
- Model: vinai/phobert-base
- Loss: CrossEntropyLoss (standard)
- No data augmentation
- No class weights
- Epochs: 10 (with early stopping patience=2)

**Purpose:**
This baseline model serves as a reference point for comparing with enhanced versions.